# 02. Road Backbone Analysis & Foundation

This notebook provides a detailed walkthrough of the **Backbone Foundation** creation process. We transform raw geospatial road data into a discrete point-based network enriched with traffic intensity, electrical capacity, and proximity to existing infrastructure.

## Objectives
1. **Discretize** LineString geometries from KMZ files into equidistant points (every 200m).
2. **Map Traffic** intensity from high-resolution road segments to our backbone points.
3. **Analyze Proximity** to existing EV charging stations and gas stations.
4. **Visualize Gaps** in the current network to identify optimal locations for new ultra-fast chargers.

In [ ]:
!git clone https://github.com/JJR9903/Iberdrola-Datathon.git

In [ ]:
cd "/content/Iberdrola-Datathon"

In [1]:
import geopandas as gpd
import pandas as pd
import folium
import os
import numpy as np
import tomllib
import sys
import polars as pl
import statsmodels.api as sm
import pmdarima as pm
from datetime import date
from dateutil.relativedelta import relativedelta



# Change directory to root
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

# Add scripts directory to path to allow module imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from scripts.sync_cloud import sync_standardized_data
from scripts.processing.create_backbone_foundation import (
    discretize_backbone_roads,
    map_traffic_to_points,
    assign_nearest_charging_stations,
    assign_nearest_gas_stations,
    assign_grid_capacity
)


In [2]:
# 1. Sync data (Ensures silver layer is present)
sync_standardized_data()


=== Iberdrola Datathon: Cloud Data Sync ===

 - Downloading roads.parquet from cloud...
 - Downloading traffic.parquet from cloud...
 - Downloading chargers.parquet from cloud...
 - Downloading electric_capacity.parquet from cloud...
 - Downloading vehicle_registrations.parquet from cloud...
 - Downloading gas_stations.parquet from cloud...

✅ Sync complete: 6 succeeded, 0 failed.


True

In [3]:
# 2. Load Config
with open('config.toml', 'rb') as f:
    config = tomllib.load(f)
params = config['steps']['backbone_foundation']


In [4]:
# 3. Load Standardized Data

print("\n🚀 Loading Standardized Data...")
gdf_roads = gpd.read_parquet(params['roads_path'])
gdf_traffic = gpd.read_parquet(params['traffic_path'])
gdf_chargers = gpd.read_parquet(params['chargers_path'])
gdf_gas = gpd.read_parquet(params['gas_stations_path'])
gdf_capacity = gpd.read_parquet(params['capacity_path'])
pl_registrations = pl.read_parquet('data/standardized/vehicle_registrations.parquet')

print(f" - Roads: {len(gdf_roads)} segments")
print(f" - Traffic: {len(gdf_traffic)} segments")
print(f" - Chargers: {len(gdf_chargers)} sites")
print(f" - Gas Stations: {len(gdf_gas)} sites")
print(f" - Electric Capacity: {len(gdf_capacity)} sites")
print(f" - Vehicle Registrations: {len(pl_registrations)}")

print("\n✅ Data loaded correctly.")


🚀 Loading Standardized Data...
 - Roads: 1591 segments
 - Traffic: 337976 segments
 - Chargers: 1298 sites
 - Gas Stations: 12204 sites
 - Electric Capacity: 2193 sites
 - Vehicle Registrations: 12969460

✅ Data loaded correctly.


In [5]:
os.getcwd()

'/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon'

visual inspection of the road dataset

In [6]:
gdf_roads.head()

,road_id,road_name,road_type,length_m,geometry
0,ID_190000,A-7S,Autopista peaje,5322.812053,"LINESTRING Z (332545.402 4043245.216 60.099, 3..."
1,ID_190001,A-7S,Autopista libre\Autovía,450.830611,"LINESTRING Z (327350.883 4042557.782 51.345, 3..."
2,ID_190002,A-7S,Autopista libre\Autovía,22389.956273,"LINESTRING Z (326926.286 4042684.128 57.736, 3..."
3,ID_190003,A-7S,Autopista peaje,2587.326470,"LINESTRING Z (308321.898 4034405.237 35.265, 3..."
4,ID_190004,A-7S,Multicarril,20452.485224,"LINESTRING Z (305774.262 4034160.476 42.619, 3..."


visual inspection of the traffic dataset

In [7]:
gdf_traffic.head()

,traffic_segment_id,geometry,total_20240331,short_20240331,medio_20240331,total_20240824,short_20240824,medio_20240824,total_20240827,short_20240827,medio_20240827,total_20241016,short_20241016,medio_20241016,total_20241019,short_20241019,medio_20241019,total_max,short_max,medio_max
0,431620080527,"LINESTRING (828792.863 4546293.054, 828793.558...",11979.36,1157.48,6033.28,16396.96,1881.43,8791.87,12653.77,1455.42,7159.17,8617.87,988.21,5578.68,11012.58,1010.20,7540.35,16396.96,1881.43,8791.87
1,450380176750,"LINESTRING (421324.886 4446687.458, 421311.987...",1814.23,1267.17,512.25,2068.41,1546.80,440.46,2286.39,1707.01,563.55,2862.48,1961.31,891.49,2412.32,1773.52,622.02,2862.48,1961.31,891.49
2,450410188347,"LINESTRING (408328.106 4452615.976, 408335.5 4...",2206.83,1848.81,321.93,2057.66,1809.55,214.73,3249.74,2717.22,508.92,4237.48,3402.05,819.92,3790.37,3204.91,559.32,4237.48,3402.05,819.92
3,451210187556,"LINESTRING (462769.447 4420963.895, 462775.306...",12426.95,665.20,6615.70,3345.17,801.43,1659.74,3973.91,784.30,2606.89,3188.89,853.67,2028.52,3903.08,842.49,2766.34,12426.95,853.67,6615.70
4,462110000438,"LINESTRING (743184.14 4314154.43, 743242.28 43...",21933.22,172.69,17106.84,44760.92,242.87,33344.83,26237.97,269.57,20655.05,14154.96,222.97,11740.40,23880.43,221.92,20112.69,44760.92,269.57,33344.83


visual inspection of the chargers dataset

In [8]:
gdf_chargers.head()

,site_id,site_name,city,province,max_power_kw,geometry,charger_count
0,0ABYOR885E4US6RFRJFV,TotalEnergies - Hotel Mirasierra,None,None,150.0,POINT (451056.608 4559802.443),4
1,0MSSBLUV4UTE7AE2J1B6,"Repsol ES, CRED Calahorra M.I.",None,None,120.0,POINT (584948.394 4682782.607),2
2,0OW3YBKDKJUTPB7ALJGO,L-VielhaEMijaran-005,None,None,110.0,POINT (811005.244 4735727.009),2
3,0S9SNBBYAX5TI0YO5JKW,"Repsol ES, La Junquera",None,None,180.0,POINT (984035.631 4710815.85),2
4,0U2UXOGZDVMXLO61BBCZ,"Repsol ES, CRED Lopidana Dcho",None,None,150.0,POINT (523465.844 4747233.793),2


visual inspection of the gas stations dataset

In [9]:
gdf_gas.head()

,station_id,city,province,geometry
0,4375,Abengibre,ALBACETE,POINT (626121.558 4341254.741)
1,5122,Alatoz,ALBACETE,POINT (643017.084 4329218.999)
2,12054,Albacete,ALBACETE,POINT (601062.716 4323495.143)
3,10765,Albacete,ALBACETE,POINT (597999.676 4315794.866)
4,4438,Albacete,ALBACETE,POINT (599900.247 4317156.738)


visual inspection of the vehicle registrations dataset

In [10]:
pl_registrations.head()

date,brand,propulsion
date,str,str
2015-01-02,"""VOLKSWAGEN""","""Diesel"""
2015-01-02,"""VOLKSWAGEN""","""Diesel"""
2015-01-02,"""LEXUS""","""Gasolina"""
2015-01-02,"""BMW""","""Diesel"""
2015-01-02,"""AUDI""","""Diesel"""


visual inspection of the electric grid capacity dataset

In [11]:
gdf_capacity.head()

,substation,grid_operator,capacity_kw,company,province,city,row_id,geometry
0,58038225,R1-299,0.0,Endesa,4,4079,R1-299_58038225,POINT (535771.67 4074262.07)
1,58037946,R1-299,0.0,Endesa,4,4903,R1-299_58037946,POINT (528430.08 4072234.8)
2,533105218,R1-299,0.0,Endesa,11,11008,R1-299_533105218,POINT (279539.31 4010543.81)
3,531021233,R1-299,3090.0,Endesa,11,11014,R1-299_531021233,POINT (224745.22 4020492.76)
4,532628547,R1-299,0.0,Endesa,11,11020,R1-299_532628547,POINT (220439.66 4061203.29)


# Step 1: EV registrations forecast

In [12]:
pl_registrations = pl_registrations.filter(
   ( pl.col('date') >= date(2015, 1, 1) ) & (pl.col('date') < date(2026, 1, 1)) 
)

In [13]:
non_ev_registrations = pl_registrations.filter(pl.col('propulsion') != 'Eléctrico').group_by(
    pl.col("date").dt.truncate("1mo")
).agg(
    pl.col('brand').count().alias('registrations')    
).sort("date").to_pandas().set_index('date')

ev_registrations = pl_registrations.filter(pl.col('propulsion') == 'Eléctrico').group_by(
    pl.col("date").dt.truncate("1mo")
).agg(
    pl.col('brand').count().alias('registrations')
).sort("date").to_pandas().set_index('date')


## step 1.2 EV forecast

In [14]:
# Selección automática del modelo con Auto-ARIMA
auto_EV_model = pm.auto_arima(np.log(ev_registrations['registrations']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=False,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)


# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_EV = auto_EV_model.order
best_seasonal_order_EV = auto_EV_model.seasonal_order

print(f"using a SARIMAX with order={best_order_EV} & seasonal_order={best_seasonal_order_EV}")

EV_model = sm.tsa.statespace.SARIMAX(
    np.log(ev_registrations['registrations']), 
    order=best_order_EV, 
    seasonal_order=best_seasonal_order_EV
)
EV_results = EV_model.fit(disp=False)

forecast_steps = 24
EV_model_forecast = EV_results.get_forecast(steps=forecast_steps)

EV_model_forecast = (EV_model_forecast.summary_frame(alpha=0.5))
EV_fc_mean = round(np.exp(EV_model_forecast['mean']))

# Crear DataFrame para el pronóstico
last_date = ev_registrations.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'registrations': EV_fc_mean
}, index=forecast_dates)


ev_registrations = pd.concat([ev_registrations, df_ev_forecast]).rename(columns={'registrations':'ev_registrations'})

ev_registrations.tail(12)

using a SARIMAX with order=(1, 0, 2) & seasonal_order=(1, 0, 1, 12)


/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rin

,ev_registrations
2027-01-01,10993.0
2027-02-01,12314.0
2027-03-01,15170.0
2027-04-01,11167.0
2027-05-01,13312.0
2027-06-01,16299.0
2027-07-01,14224.0
2027-08-01,11609.0
2027-09-01,16695.0
2027-10-01,16779.0


## 1.3 Non EV Registrations

In [15]:
# Selección automática del modelo con Auto-ARIMA
auto_nEV_model = pm.auto_arima(np.log(non_ev_registrations['registrations']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=False,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)


# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_nEV = auto_nEV_model.order
best_seasonal_order_nEV = auto_nEV_model.seasonal_order

print(f"using a SARIMAX with order={best_order_nEV} & seasonal_order={best_seasonal_order_nEV} with drift")

nEV_model = sm.tsa.statespace.SARIMAX(
    np.log(non_ev_registrations['registrations']), 
    trend='c' if auto_nEV_model.with_intercept==True else 'n',
    order=best_order_nEV, 
    seasonal_order=best_seasonal_order_nEV
)
nEV_results = nEV_model.fit(disp=False)

forecast_steps = 24
nEV_model_forecast = nEV_results.get_forecast(steps=forecast_steps)

nEV_model_forecast = (nEV_model_forecast.summary_frame(alpha=0.5))
nEV_fc_mean = round(np.exp(nEV_model_forecast['mean']))

# Crear DataFrame para el pronóstico
last_date = non_ev_registrations.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'registrations': nEV_fc_mean
}, index=forecast_dates)


non_ev_registrations = pd.concat([non_ev_registrations, df_ev_forecast]).rename(columns={'registrations':'non_ev_registrations'})

non_ev_registrations

using a SARIMAX with order=(0, 0, 2) & seasonal_order=(0, 0, 2, 12) with drift


/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/Users/juanjose/Library/CloudStorage/GoogleDrive-jj.rincon@student.ie.edu/My Drive/Iberdrola Datathon/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:1009: UserWarning: Non-invertible starting seasonal moving average Using zeros as starting parameters.
  warn('Non-invertible starting seasonal moving average'


,non_ev_registrations
2015-01-01,71240.0
2015-02-01,90257.0
2015-03-01,115720.0
2015-04-01,86170.0
2015-05-01,97666.0
...,...
2027-08-01,87452.0
2027-09-01,90111.0
2027-10-01,91402.0
2027-11-01,90980.0


In [16]:
vehicle_registrations = ev_registrations.merge(non_ev_registrations,how='inner',left_index=True, right_index=True)
vehicle_registrations

,ev_registrations,non_ev_registrations
2015-01-01,55.0,71240.0
2015-02-01,51.0,90257.0
2015-03-01,162.0,115720.0
2015-04-01,111.0,86170.0
2015-05-01,118.0,97666.0
...,...,...
2027-08-01,11609.0,87452.0
2027-09-01,16695.0,90111.0
2027-10-01,16779.0,91402.0
2027-11-01,17671.0,90980.0


In [17]:
vehicle_registrations_total = pd.DataFrame({
    "total_ev_registrations": [vehicle_registrations["ev_registrations"].sum()],
    "total_non_ev_registrations": [vehicle_registrations["non_ev_registrations"].sum()],
})

vehicle_registrations_total["pct_ev_registrations"] = (
    vehicle_registrations_total["total_ev_registrations"] /
    (vehicle_registrations_total["total_ev_registrations"] + vehicle_registrations_total["total_non_ev_registrations"])
)
vehicle_registrations_total

,total_ev_registrations,total_non_ev_registrations,pct_ev_registrations
0,664995.0,14626597.0,0.043488


In [18]:
vehicle_registrations_2027 = pd.DataFrame({
    "total_ev_registrations": [vehicle_registrations.loc[vehicle_registrations.index>='2027-01-01']["ev_registrations"].sum()],
    "total_non_ev_registrations": [vehicle_registrations.loc[vehicle_registrations.index>='2027-01-01']["non_ev_registrations"].sum()],
})

vehicle_registrations_2027["pct_ev_registrations"] = (
    vehicle_registrations_2027["total_ev_registrations"] /
    (vehicle_registrations_2027["total_ev_registrations"] + vehicle_registrations_2027["total_non_ev_registrations"])
)
vehicle_registrations_2027

,total_ev_registrations,total_non_ev_registrations,pct_ev_registrations
0,176583.0,1093526.0,0.13903


## Step 1: Discretization
The raw road backbone is provided as high-level `LineString` geometries in a KMZ file. To perform precise point-based analysis (like calculating the distance to the exact nearest charger), we convert these lines into points sampled every 200 meters.

**Assumptions:**
*   A **200m interval** provides a good balance between spatial resolution and computational efficiency.
*   We preserve the original `backbone_id` to maintain traceability to the source data.

In [19]:
params['sampling_interval_m']

200

In [20]:
gdf_points = discretize_backbone_roads(
    gdf_roads,
    sampling_interval_m=params['sampling_interval_m']
)
print(f"Generated {len(gdf_points)} points along the backbone.")
gdf_points.head()

 - Discretizing backbones into points (Interval=200m)...
Generated 147288 points along the backbone.


,backbone_id,road_name,road_type,length_m,geometry,m_ref,point_idx,point_id
0,ID_190000,A-7S,Autopista peaje,5322.812053,POINT Z (332545.402 4043245.216 60.099),0.0,0,ID_190000_0
1,ID_190000,A-7S,Autopista peaje,5322.812053,POINT Z (332347.326 4043217.681 69.7),200.0,1,ID_190000_1
2,ID_190000,A-7S,Autopista peaje,5322.812053,POINT Z (332147.851 4043216.941 68.397),400.0,2,ID_190000_2
3,ID_190000,A-7S,Autopista peaje,5322.812053,POINT Z (331953.2 4043260.901 71.507),600.0,3,ID_190000_3
4,ID_190000,A-7S,Autopista peaje,5322.812053,POINT Z (331763.559 4043324.239 74.492),800.0,4,ID_190000_4


### Visualization: Base Point Network
Below we see the result of the discretization. Each red dot represents a sampling point where we will later attach traffic and infrastructure data.

In [ ]:
m0 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
gdf_plot = gdf_points.to_crs(4326)

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=1,
        color='red',
        fill=True
    ).add_to(m0)
m0

## Step 2: Traffic Intensity Mapping
Traffic data is usually collected on specific road segments. We map this data to our backbone points using a spatial buffer.

**Assumptions & Logic:**
*   **Buffer Radius (50m)**: We look for road segments within 50m of each point.
*   **Neighbor Validation**: To avoid "cross-talk" from overpasses or intersecting roads, we only accept a segment if it intersects at least two consecutive points on the same backbone. This ensures the segment actually runs longitudinal to the backbone.

In [24]:
params['traffic_columns']

['total_max', 'short_max', 'medio_max']

In [25]:
params['buffer_radius_m']

50

In [26]:
params['traffic_columns']

['total_max', 'short_max', 'medio_max']

In [27]:
gdf_points = map_traffic_to_points(
    gdf_points,
    gdf_traffic,
    params['traffic_columns'],
    params['buffer_radius_m']
)
gdf_points[params['traffic_columns']].describe()


 - Mapping traffic columns ['total_max', 'short_max', 'medio_max'] (Buffer=50m)...
   Mapping columns: ['total_max', 'short_max', 'medio_max']


,total_max,short_max,medio_max
count,147288.000000,147288.000000,147288.000000
mean,32861.475829,7600.897533,12633.115880
std,47627.030784,19795.059084,20326.373441
min,0.000000,0.000000,0.000000
25%,288.410000,119.810000,39.750000
50%,12477.780000,1769.810000,4757.690000
75%,47696.297500,5890.140000,15662.060000
max,628705.660000,546100.160000,318188.120000


### Map 1: Road & Traffic
Points are colored based on their traffic intensity (`total_max`). Green indicates light traffic, yellow mid traffic, and red heavy traffic.

In [ ]:
import branca.colormap as cm

m1 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
colormap = cm.LinearColormap(['green', 'yellow', 'red'], vmin=0, vmax=gdf_points['total_max'].quantile(0.95))
colormap.caption = 'Traffic Intensity (Total Max)'
colormap.add_to(m1)

gdf_plot = gdf_points.to_crs(4326)
for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=colormap(row['total_max']),
        fill=True,
        tooltip=f"Traffic: {row['total_max']}"
    ).add_to(m1)
m1

## Step 3: Infrastructure Proximity
We now calculate the distance to the nearest existing EV charging station and gas station.

**Assumptions:**
*   The nearest charger is identified using a spatial `nearest` join.
*   Distances are calculated in meters using the metric CRS (EPSG:3042).

In [28]:
params['max_distance_proximity']

100000

In [29]:
gdf_points = assign_nearest_charging_stations(
    gdf_points = gdf_points,
    gdf_chargers = gdf_chargers,
    max_distance = params['max_distance_proximity']
)

 - Assigning nearest charging stations (MaxDist=100000)...


In [30]:
gdf_points = assign_nearest_gas_stations(
    gdf_points = gdf_points,
    gdf_gas = gdf_gas,
    max_distance = params['max_distance_proximity']
)
print("Proximity analysis completed.")

 - Assigning nearest gas stations (MaxDist=100000)...
Proximity analysis completed.


### Map 2: Road & Current EV Chargers
This map highlights the areas with existing infrastructure. Points are colored by distance to the nearest charger (Green = Close, Red = Far).

In [ ]:
gdf_plot = gdf_points.to_crs(4326)

import branca.colormap as cm

m2 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
dist_colormap = cm.LinearColormap(['green', 'yellow', 'red'], vmin=0, vmax=max(gdf_plot['dist_charger_m']))
dist_colormap.caption = 'Distance to Nearest Charger (m)'
dist_colormap.add_to(m2)

# Explicitly fill NaN values in 'dist_charger_m' before iteration
gdf_plot['dist_charger_m'] = gdf_plot['dist_charger_m'].fillna(max(gdf_plot['dist_charger_m']))

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=dist_colormap(min(row['dist_charger_m'], 100000)), # Use the cleaned column directly
        fill=True
    ).add_to(m2)
m2

# Step 4: Optimization

In [31]:
from scripts.processing.optimize_grid_aware_placement import generate_smart_candidates, solve_grid_aware_optimization, report

In [32]:
ev_traffic_pct = vehicle_registrations_2027['pct_ev_registrations'].iloc[0]
print(ev_traffic_pct)

0.13902979980458371


In [33]:
need_charge_pct = gdf_points['medio_max'].sum() / gdf_points['total_max'].sum()
need_charge_pct

np.float64(0.384435438796252)

In [34]:
max_chargers_per_site =  gdf_chargers['charger_count'].max()
max_chargers_per_site

np.int64(30)

In [35]:
coverage_threshold_m = 30000
sessions_per_charger = 24
substation_threshold_m = 10000
power_per_charger_kw = 150
penalty_coverage = 1e6
penalty_supply = 1e4
penalty_grid_upgrade = 1e2

In [36]:
df_cand = generate_smart_candidates(
    gdf_backbone = gdf_points, 
    gdf_chargers = gdf_chargers, 
    gdf_gas = gdf_gas, 
    ev_traffic_pct = ev_traffic_pct, 
    need_charge_pct = need_charge_pct, 
    coverage_threshold_m = coverage_threshold_m, 
    max_chargers_per_site = max_chargers_per_site, 
    sessions_per_charger = sessions_per_charger
    )

📍 Generating Smart Candidates...


In [37]:
df_res, grid_slacks = solve_grid_aware_optimization(
        gdf_backbone = gdf_points, 
        df_cand = df_cand, 
        gdf_grid = gdf_capacity,
        ev_traffic_pct = ev_traffic_pct,
        need_charge_pct= need_charge_pct,
        coverage_threshold_m = coverage_threshold_m,
        substation_threshold_m = substation_threshold_m,
        max_chargers_per_site = max_chargers_per_site,
        sessions_per_charger = sessions_per_charger,
        power_per_charger_kw = power_per_charger_kw,
        penalty_coverage = penalty_coverage,
        penalty_supply = penalty_supply,
        penalty_grid_upgrade = penalty_grid_upgrade
    )



🧠 Formulating Grid-Aware optimization (Vars: 323823)...
   Building road constraints...
   Building grid constraints...
🚀 Solving Grid-Aware Linear Relaxation...
   Optimization Successful! (Time: 11.94s)


In [38]:
report(df_res, grid_slacks)


📊 --- GRID-AWARE OPTIMIZATION SUMMARY ---
Total Stations Recommendation = 2356
Total New Chargers = 42103
Substations requiring UPGRADES = 783
Total Grid Capacity Gap = 5020940 kW
  - Grid Upgrade Required: 1853
  - Feasible: 503
-------------------------------------------



In [39]:
gdf_capacity.loc[gdf_capacity['capacity_kw']>0]

,substation,grid_operator,capacity_kw,company,province,city,row_id,geometry
3,531021233,R1-299,3090.0,Endesa,11,11014,R1-299_531021233,POINT (224745.22 4020492.76)
21,58020973,R1-299,79090.0,Endesa,29,29007,R1-299_58020973,POINT (359235.63 4057869.8)
23,531988838,R1-299,44150.0,Endesa,29,29039,R1-299_531988838,POINT (373361.56 4084869.28)
28,518000047,R1-299,18880.0,Endesa,29,29067,R1-299_518000047,POINT (378623.38 4065512.9)
29,58002206,R1-299,15740.0,Endesa,29,29084,R1-299_58002206,POINT (309034.81 4068931.53)
...,...,...,...,...,...,...,...,...
2172,3316,R1-001,9710.0,Iberdrola,Madrid,Madrid,R1-001_3316,POINT (439099.758 4470901.031)
2174,3307,R1-001,36640.0,Iberdrola,Madrid,Madrid,R1-001_3307,POINT (440609.506 4480596.524)
2184,3514,R1-001,40590.0,Iberdrola,Murcia,Cartagena,R1-001_3514,POINT (679903.167 4167145.118)
2185,3761,R1-001,77790.0,Iberdrola,Murcia,Fuente Álamo de Murcia,R1-001_3761,POINT (661104.557 4172066.44)


In [40]:
df_res.loc[(df_res['is_open']==1)&(df_res['type']!='existing')]

,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility
1356,4383,gas,0,POINT (658248.24357504 4304489.891626588),1287,R1-001_4262,4967.091944,1,4,4,Grid Upgrade Required
1360,9035,gas,0,POINT (652793.5156635651 4313691.606281224),440,R1-001_4243,2510.186304,1,4,4,Grid Upgrade Required
1363,4399,gas,0,POINT (569652.4634162581 4321871.955985377),151,R1-001_3590,3295.281534,1,4,4,Grid Upgrade Required
1366,5282,gas,0,POINT (540286.8272060794 4311703.790309232),1530,R1-001_3591,784.996461,1,19,19,Grid Upgrade Required
1467,5123,gas,0,POINT (533322.1193997232 4345426.445918975),153,R1-001_3559,3277.660224,1,16,16,Grid Upgrade Required
...,...,...,...,...,...,...,...,...,...,...,...
13407,1500,gas,0,POINT (666060.9780256399 4618803.16866515),1485,R1-299_1750014,1517.887680,1,9,9,Grid Upgrade Required
13409,9079,gas,0,POINT (714485.7107031441 4583650.83400776),1484,R1-299_1250056,5181.799604,1,30,30,Grid Upgrade Required
13499,9085,gas,0,POINT (683959.8727885496 4639227.62049227),1487,R1-299_1250026,1349.707892,1,13,13,Grid Upgrade Required
13502,GF_ID_190375_106,greenfield,0,POINT Z (853243.0398258348 4699374.922995339 1...,132,R1-299_4244466,9971.315765,1,30,30,Grid Upgrade Required


In [41]:
df_res.loc[(df_res['type']!='existing')]

,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility
1298,4375,gas,0,POINT (626121.558326784 4341254.740725788),1532,R1-001_5086,9034.367367,0,0,0,Feasible
1299,5122,gas,0,POINT (643017.0843019226 4329218.9989114385),2193,NaN,inf,0,0,0,No Grid Access (>10km)
1300,12054,gas,0,POINT (601062.7158447192 4323495.143079704),1807,R1-001_3605,2985.186106,0,0,0,Feasible
1301,10765,gas,0,POINT (597999.6761573086 4315794.866009133),2086,R1-001_4007,368.204578,0,0,0,Grid Upgrade Required
1302,4438,gas,0,POINT (599900.2470567425 4317156.7375265155),2086,R1-001_4007,2618.714071,0,0,0,Grid Upgrade Required
...,...,...,...,...,...,...,...,...,...,...,...
13522,GF_ID_190987_484,greenfield,0,POINT Z (185510.5229180133 4661123.400129799 1...,2193,NaN,inf,0,0,0,No Grid Access (>10km)
13523,GF_ID_190987_488,greenfield,0,POINT Z (184722.25327199246 4661221.652414905 ...,2193,NaN,inf,0,0,0,No Grid Access (>10km)
13524,GF_ID_190987_490,greenfield,0,POINT Z (184338.28362916387 4661117.534633813 ...,2193,NaN,inf,0,0,0,No Grid Access (>10km)
13525,GF_ID_190987_581,greenfield,0,POINT Z (167232.9753072194 4661846.0648872135 ...,2193,NaN,inf,0,0,0,No Grid Access (>10km)


In [42]:
gdf_res = gpd.GeoDataFrame(df_res, geometry='geometry', crs=gdf_points.crs)
gdf_res.to_parquet("data/processed/grid_aware_optimized_sites.parquet")

In [43]:
gdf_res

,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility
0,0ABYOR885E4US6RFRJFV,existing,4,POINT (451056.608 4559802.443),2193,NaN,inf,0,4,0,No Grid Access (>10km)
1,0MSSBLUV4UTE7AE2J1B6,existing,2,POINT (584948.394 4682782.607),1607,R1-001_4629,1370.993323,1,30,28,Grid Upgrade Required
2,0OW3YBKDKJUTPB7ALJGO,existing,2,POINT (811005.244 4735727.009),1795,R1-299_4587714,220.657489,1,30,28,Grid Upgrade Required
3,0S9SNBBYAX5TI0YO5JKW,existing,2,POINT (984035.631 4710815.85),2193,NaN,inf,0,2,0,No Grid Access (>10km)
4,0U2UXOGZDVMXLO61BBCZ,existing,2,POINT (523465.844 4747233.793),148,R1-001_3017,2403.488143,1,4,2,Grid Upgrade Required
...,...,...,...,...,...,...,...,...,...,...,...
13522,GF_ID_190987_484,greenfield,0,POINT Z (185510.523 4661123.4 1239.044),2193,NaN,inf,0,0,0,No Grid Access (>10km)
13523,GF_ID_190987_488,greenfield,0,POINT Z (184722.253 4661221.652 1278.157),2193,NaN,inf,0,0,0,No Grid Access (>10km)
13524,GF_ID_190987_490,greenfield,0,POINT Z (184338.284 4661117.535 1308.832),2193,NaN,inf,0,0,0,No Grid Access (>10km)
13525,GF_ID_190987_581,greenfield,0,POINT Z (167232.975 4661846.065 1046.381),2193,NaN,inf,0,0,0,No Grid Access (>10km)


## Output construction

Add capacity needed

In [58]:
gdf_res['estimated_demand_kw'] = gdf_res['final_n'] * 150

Add Longitude and Latitude

In [59]:
gdf_res['latitude'] = gdf_res.geometry.y
gdf_res['longitude'] = gdf_res.geometry.x

Filtered only points with a new charging station

In [118]:
df_stations = gdf_res[(gdf_res['is_open']==1)&(gdf_res['type']!='existing')]

In [119]:
gdf_res[(gdf_res['is_open']==1)]

,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility,estimated_demand_kw,latitude,longitude
1,0MSSBLUV4UTE7AE2J1B6,existing,2,POINT (584948.394 4682782.607),1607,R1-001_4629,1370.993323,1,30,28,Grid Upgrade Required,4500,4.682783e+06,584948.394006
2,0OW3YBKDKJUTPB7ALJGO,existing,2,POINT (811005.244 4735727.009),1795,R1-299_4587714,220.657489,1,30,28,Grid Upgrade Required,4500,4.735727e+06,811005.243836
4,0U2UXOGZDVMXLO61BBCZ,existing,2,POINT (523465.844 4747233.793),148,R1-001_3017,2403.488143,1,4,2,Grid Upgrade Required,600,4.747234e+06,523465.844158
5,14OHI9EMAHFI3X4WQ1YU,existing,8,POINT (903508.229 4618428.296),1261,R1-299_4644671,793.253345,1,8,0,Feasible,1200,4.618428e+06,903508.228737
6,1BEIJQUXN6N3M30WQUCC,existing,4,POINT (236833.558 4141663.271),1448,R1-299_510006035,915.309228,1,4,0,Feasible,600,4.141663e+06,236833.557735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13407,1500,gas,0,POINT (666060.978 4618803.169),1485,R1-299_1750014,1517.887680,1,9,9,Grid Upgrade Required,1350,4.618803e+06,666060.978026
13409,9079,gas,0,POINT (714485.711 4583650.834),1484,R1-299_1250056,5181.799604,1,30,30,Grid Upgrade Required,4500,4.583651e+06,714485.710703
13499,9085,gas,0,POINT (683959.873 4639227.62),1487,R1-299_1250026,1349.707892,1,13,13,Grid Upgrade Required,1950,4.639228e+06,683959.872789
13502,GF_ID_190375_106,greenfield,0,POINT Z (853243.04 4699374.923 1298.935),132,R1-299_4244466,9971.315765,1,30,30,Grid Upgrade Required,4500,4.699375e+06,853243.039826


Add Loc Id for every station

In [ ]:
df_stations['location_id'] = [f"IBE_{i:03d}" for i in range(1, len(df_stations) + 1)]

In [121]:
gdf_capacity

,substation,grid_operator,capacity_kw,company,province,city,row_id,geometry
0,58038225,R1-299,0.0,Endesa,4,4079,R1-299_58038225,POINT (535771.67 4074262.07)
1,58037946,R1-299,0.0,Endesa,4,4903,R1-299_58037946,POINT (528430.08 4072234.8)
2,533105218,R1-299,0.0,Endesa,11,11008,R1-299_533105218,POINT (279539.31 4010543.81)
3,531021233,R1-299,3090.0,Endesa,11,11014,R1-299_531021233,POINT (224745.22 4020492.76)
4,532628547,R1-299,0.0,Endesa,11,11020,R1-299_532628547,POINT (220439.66 4061203.29)
...,...,...,...,...,...,...,...,...
2188,3666,R1-001,0.0,Iberdrola,Murcia,Totana,R1-001_3666,POINT (636802.128 4180915.37)
2189,5499,R1-001,0.0,Iberdrola,Murcia,Pliego,R1-001_5499,POINT (633296.427 4201927.511)
2190,9693,R1-001,0.0,Iberdrola,Navarra,Valtierra,R1-001_9693,POINT (610471.74 4673619.227)
2191,4710,R1-001,0.0,Iberdrola,Navarra,Huarte/Uharte,R1-001_4710,POINT (614245.21 4743705.687)


In [122]:
df_stations

,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility,estimated_demand_kw,latitude,longitude,location_id
1356,4383,gas,0,POINT (658248.244 4304489.892),1287,R1-001_4262,4967.091944,1,4,4,Grid Upgrade Required,600,4.304490e+06,658248.243575,IBE_01
1360,9035,gas,0,POINT (652793.516 4313691.606),440,R1-001_4243,2510.186304,1,4,4,Grid Upgrade Required,600,4.313692e+06,652793.515664,IBE_02
1363,4399,gas,0,POINT (569652.463 4321871.956),151,R1-001_3590,3295.281534,1,4,4,Grid Upgrade Required,600,4.321872e+06,569652.463416,IBE_03
1366,5282,gas,0,POINT (540286.827 4311703.79),1530,R1-001_3591,784.996461,1,19,19,Grid Upgrade Required,2850,4.311704e+06,540286.827206,IBE_04
1467,5123,gas,0,POINT (533322.119 4345426.446),153,R1-001_3559,3277.660224,1,16,16,Grid Upgrade Required,2400,4.345426e+06,533322.119400,IBE_05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13407,1500,gas,0,POINT (666060.978 4618803.169),1485,R1-299_1750014,1517.887680,1,9,9,Grid Upgrade Required,1350,4.618803e+06,666060.978026,IBE_1235
13409,9079,gas,0,POINT (714485.711 4583650.834),1484,R1-299_1250056,5181.799604,1,30,30,Grid Upgrade Required,4500,4.583651e+06,714485.710703,IBE_1236
13499,9085,gas,0,POINT (683959.873 4639227.62),1487,R1-299_1250026,1349.707892,1,13,13,Grid Upgrade Required,1950,4.639228e+06,683959.872789,IBE_1237
13502,GF_ID_190375_106,greenfield,0,POINT Z (853243.04 4699374.923 1298.935),132,R1-299_4244466,9971.315765,1,30,30,Grid Upgrade Required,4500,4.699375e+06,853243.039826,IBE_1238


Add road name

In [123]:
df_stations_indexed = df_stations.set_index('location_id')

df_stations_with_road = gpd.sjoin_nearest(
    df_stations_indexed,
    gdf_points[['road_name', 'geometry']],
    how='left', 
)

df_stations_with_road = df_stations_with_road.loc[~df_stations_with_road.index.duplicated(keep='first')]

df_stations_with_road.rename(columns={'road_name': 'nearest_road_name'}, inplace=True)

df_stations = df_stations.set_index('location_id')
df_stations['route_segment'] = df_stations_with_road['nearest_road_name']
df_stations = df_stations.reset_index() 


In [124]:
df_stations

,location_id,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility,estimated_demand_kw,latitude,longitude,route_segment
0,IBE_01,4383,gas,0,POINT (658248.244 4304489.892),1287,R1-001_4262,4967.091944,1,4,4,Grid Upgrade Required,600,4.304490e+06,658248.243575,A-31
1,IBE_02,9035,gas,0,POINT (652793.516 4313691.606),440,R1-001_4243,2510.186304,1,4,4,Grid Upgrade Required,600,4.313692e+06,652793.515664,A-31
2,IBE_03,4399,gas,0,POINT (569652.463 4321871.956),151,R1-001_3590,3295.281534,1,4,4,Grid Upgrade Required,600,4.321872e+06,569652.463416,N-430
3,IBE_04,5282,gas,0,POINT (540286.827 4311703.79),1530,R1-001_3591,784.996461,1,19,19,Grid Upgrade Required,2850,4.311704e+06,540286.827206,N-430
4,IBE_05,5123,gas,0,POINT (533322.119 4345426.446),153,R1-001_3559,3277.660224,1,16,16,Grid Upgrade Required,2400,4.345426e+06,533322.119400,N-310
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1234,IBE_1235,1500,gas,0,POINT (666060.978 4618803.169),1485,R1-299_1750014,1517.887680,1,9,9,Grid Upgrade Required,1350,4.618803e+06,666060.978026,N-232
1235,IBE_1236,9079,gas,0,POINT (714485.711 4583650.834),1484,R1-299_1250056,5181.799604,1,30,30,Grid Upgrade Required,4500,4.583651e+06,714485.710703,N-232
1236,IBE_1237,9085,gas,0,POINT (683959.873 4639227.62),1487,R1-299_1250026,1349.707892,1,13,13,Grid Upgrade Required,1950,4.639228e+06,683959.872789,A-23
1237,IBE_1238,GF_ID_190375_106,greenfield,0,POINT Z (853243.04 4699374.923 1298.935),132,R1-299_4244466,9971.315765,1,30,30,Grid Upgrade Required,4500,4.699375e+06,853243.039826,N-260


In [125]:
capacity_usefull_cols = gdf_capacity[['company','row_id','capacity_kw']]

df_stations = df_stations.merge(
    capacity_usefull_cols,
    left_on = 'substation_id',
    right_on = 'row_id',
    how = 'left'
    )

df_stations.head()

,location_id,site_id,type,initial_n,geometry,substation_pos_idx,substation_id,dist_to_grid,is_open,final_n,added_chargers,grid_feasibility,estimated_demand_kw,latitude,longitude,route_segment,company,row_id,capacity_kw
0,IBE_01,4383,gas,0,POINT (658248.244 4304489.892),1287,R1-001_4262,4967.091944,1,4,4,Grid Upgrade Required,600,4.304490e+06,658248.243575,A-31,Iberdrola,R1-001_4262,0.0
1,IBE_02,9035,gas,0,POINT (652793.516 4313691.606),440,R1-001_4243,2510.186304,1,4,4,Grid Upgrade Required,600,4.313692e+06,652793.515664,A-31,Iberdrola,R1-001_4243,0.0
2,IBE_03,4399,gas,0,POINT (569652.463 4321871.956),151,R1-001_3590,3295.281534,1,4,4,Grid Upgrade Required,600,4.321872e+06,569652.463416,N-430,Iberdrola,R1-001_3590,0.0
3,IBE_04,5282,gas,0,POINT (540286.827 4311703.79),1530,R1-001_3591,784.996461,1,19,19,Grid Upgrade Required,2850,4.311704e+06,540286.827206,N-430,Iberdrola,R1-001_3591,0.0
4,IBE_05,5123,gas,0,POINT (533322.119 4345426.446),153,R1-001_3559,3277.660224,1,16,16,Grid Upgrade Required,2400,4.345426e+06,533322.119400,N-310,Iberdrola,R1-001_3559,0.0


In [126]:
subs_demand = df_stations.groupby('substation_id')[['estimated_demand_kw','capacity_kw']].agg({'estimated_demand_kw':'sum','capacity_kw':'max'}).reset_index()

subs_demand['ratio'] = subs_demand['capacity_kw']/subs_demand['estimated_demand_kw']
subs_demand['grid_status'] = np.where(
    (subs_demand['ratio'] >1.2)|(subs_demand['ratio']==0), 'Congested',
    np.where(subs_demand['ratio']>1, 'Moderate', 'Sufficient')
)

df_final = df_stations.merge(
    subs_demand[['substation_id','ratio','grid_status']],
    on = 'substation_id',
    how = 'left'
    )

In [128]:
file_2 = df_final[['location_id','latitude','longitude','route_segment','final_n','grid_status']]
file_2.rename(columns = {'final_n': 'n_chargers_proposed'},inplace = True)
file_2.head()

,location_id,latitude,longitude,route_segment,n_chargers_proposed,grid_status
0,IBE_01,4.304490e+06,658248.243575,A-31,4,Congested
1,IBE_02,4.313692e+06,652793.515664,A-31,4,Congested
2,IBE_03,4.321872e+06,569652.463416,N-430,4,Congested
3,IBE_04,4.311704e+06,540286.827206,N-430,19,Congested
4,IBE_05,4.345426e+06,533322.119400,N-310,16,Congested


In [129]:
file_2.to_csv('data/outputs/File 2.csv')

In [136]:
file_3 = df_final[['latitude','longitude','route_segment','company','estimated_demand_kw','grid_status']]
file_3 = file_3[file_3['grid_status']!= 'Sufficent']
file_3['bottleneck_id'] = [f"FRIC_{i:03d}" for i in range(1, len(file_3) + 1)]
file_3 = file_3[['bottleneck_id','latitude','longitude','route_segment','company','estimated_demand_kw','grid_status']]
file_3.rename(columns = {'company': 'distributor_network'}, inplace = True )
file_3.head()

,bottleneck_id,latitude,longitude,route_segment,distributor_network,estimated_demand_kw,grid_status
0,FRIC_001,4.304490e+06,658248.243575,A-31,Iberdrola,600,Congested
1,FRIC_002,4.313692e+06,652793.515664,A-31,Iberdrola,600,Congested
2,FRIC_003,4.321872e+06,569652.463416,N-430,Iberdrola,600,Congested
3,FRIC_004,4.311704e+06,540286.827206,N-430,Iberdrola,2850,Congested
4,FRIC_005,4.345426e+06,533322.119400,N-310,Iberdrola,2400,Congested


In [140]:
file_3.to_csv('data/outputs/File 3.csv')

In [139]:
file_1 = pd.DataFrame({
    'total_proposed_stations': [len(file_2)],
    'total_exisiting_stations_baseline': [len(gdf_chargers)],
    'total_friction_points': [len(file_3)],
    'total_ev_projected_2027' :  [vehicle_registrations_total['total_ev_registrations'].item()]
})
file_1

,total_proposed_stations,total_exisiting_stations_baseline,total_friction_points,total_ev_projected_2027
0,1239,1298,1239,664995.0


In [141]:
file_1.to_csv('data/outputs/File 1.csv')

In [143]:

# 1. Map Colors
color_map = {
    'Sufficient': 'green',
    'Moderate': 'yellow',
    'Congested': 'red'
}

# 2. Coordinate Transformation (Optional: only if coordinates are still in projected meters)
# Assuming source is EPSG:3042 (Spain UTM) and converting to WGS84 for Folium
if 'geometry' in file_2.columns and isinstance(file_2, gpd.GeoDataFrame):
    df_map = file_2.to_crs(epsg=4326)
else:
    # If it's a CSV with X/Y columns in meters, create GeoDataFrame first
    gdf = gpd.GeoDataFrame(
        file_2, 
        geometry=gpd.points_from_xy(file_2.longitude, file_2.latitude),
        crs="EPSG:3042"
    )
    df_map = gdf.to_crs(epsg=4326)

# 3. Initialize Map
m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')

# 4. Add Points to Map
for _, row in df_map.iterrows():
    # Use Tooltip for hover information
    tooltip_html = f"""
        <b>Route Segment:</b> {row['route_segment']}<br>
        <b>Proposed Chargers:</b> {row['n_chargers_proposed']}<br>
        <b>Grid Status:</b> {row['grid_status']}<br>
        <b>Coordinates:</b> ({row.geometry.y:.5f}, {row.geometry.x:.5f})
    """
    
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color=color_map.get(row['grid_status'], 'gray'),
        fill=True,
        fill_color=color_map.get(row['grid_status'], 'gray'),
        fill_opacity=0.7,
        tooltip=tooltip_html  # Shows on hover
    ).add_to(m)

# Display Map
m.save("grid_status_map.html") # Or just 'm' in a notebook

m


